<a href="https://colab.research.google.com/github/Meghii/sql-business-analysis/blob/main/chinook_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
import sqlite3 # loads Python’s built-in module for working with SQLite databases
import pandas as pd

con = sqlite3.connect('Chinook_Sqlite.sqlite') # opens the database file and returns a connection object, saved in the variable con

## `sqlite_master` — a special built-in table that lists all the tables in the database.

In [57]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", con)

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


View the First five customer details from Customer table.

To view upto a paticular No.of rows, use `LIMIT`

In [58]:
pd.read_sql_query("SELECT * FROM Customer LIMIT 5;", con)

,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,2,Leonie,Köhler,None,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,+49 0711 2842222,None,leonekohler@surfeu.de,5
2,3,François,Tremblay,None,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,None,ftremblay@gmail.com,3
3,4,Bjørn,Hansen,None,Ullevålsveien 14,Oslo,None,Norway,0171,+47 22 44 22 22,None,bjorn.hansen@yahoo.no,4
4,5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,None,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


Since Colab runs the cell as Python, and Python doesn’t understand SELECT, so it reports SyntaxError: invalid syntax

**Fix**: wrap the SQL in pd.read_sql and pass the connection

*   The triple quotes `""" ... """` let you write the SQL over several lines as a Python string.
*   `pd.read_sql(query, con) `sends that string to the database through your connection con and returns the result as a DataFrame.
*   The last line, `df_revenue`, displays the table.

**Q1.** Which countries bring in the most revenue?

The USA is the biggest market, followed by Canada and France.

In [59]:
query = """
SELECT BillingCountry, SUM(Total) AS revenue
FROM Invoice
GROUP BY BillingCountry
ORDER BY revenue DESC
LIMIT 10;
"""
df_revenue = pd.read_sql(query, con)
df_revenue

,BillingCountry,revenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48
5,United Kingdom,112.86
6,Czech Republic,90.24
7,Portugal,77.24
8,India,75.26
9,Chile,46.62


**Q2.** Who are the top 10 customers by total spend?

The top 10 customers each spent between 42 & 50, led by Helena Holý (Czech Republic) at $49.62, with the USA appearing most often (4 of 10).

In SQLite, `**|| ' ' ||**` joins text together (like + for strings in Python)

In [60]:
query = """
SELECT c.CustomerId, c.FirstName || ' ' || c.LastName AS customer, c.Country, SUM(i.Total) AS total_spent
FROM Customer AS c
JOIN Invoice AS i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId
ORDER BY total_spent DESC
LIMIT 10;
"""
df_total_spent = pd.read_sql(query, con)
df_total_spent

,CustomerId,customer,Country,total_spent
0,6,Helena Holý,Czech Republic,49.62
1,26,Richard Cunningham,USA,47.62
2,57,Luis Rojas,Chile,46.62
3,45,Ladislav Kovács,Hungary,45.62
4,46,Hugh O'Reilly,Ireland,45.62
5,24,Frank Ralston,USA,43.62
6,28,Julia Barnett,USA,43.62
7,37,Fynn Zimmermann,Germany,43.62
8,7,Astrid Gruber,Austria,42.62
9,25,Victor Stevens,USA,42.62


**Q3.** What's the average invoice value in each country?

Chile has the highest average invoice at 6.66, followed by Ireland and Hungary. And the top 10 countries all fall within a narrow 5.66 to 6.66 range.

In [61]:
query = """
SELECT BillingCountry, ROUND(AVG(Total),2) AS avg_revenue
FROM Invoice
GROUP BY BillingCountry
ORDER BY avg_revenue DESC
LIMIT 10;
"""
df_avg_revenue = pd.read_sql(query, con)
df_avg_revenue

,BillingCountry,avg_revenue
0,Chile,6.66
1,Ireland,6.52
2,Hungary,6.52
3,Czech Republic,6.45
4,Austria,6.09
5,Finland,5.95
6,Netherlands,5.80
7,India,5.79
8,USA,5.75
9,Norway,5.66


### **Joins across tables**

**Q4.** Which genres sell the most (by revenue)?

Rock is the top-earning genre (about 35% of the store’s total revenue), followed by Latin, Metal & Alternative & Punk.

Those top four genres bring in roughly 74% of revenue.

In [62]:
query="""
SELECT g.Name AS genre, SUM(il.UnitPrice * il.Quantity) AS revenue
FROM InvoiceLine il
JOIN Track t ON il.TrackId = t.TrackId
JOIN Genre g ON t.GenreId = g.GenreId
GROUP BY g.Name
ORDER BY revenue DESC
LIMIT 10;
"""
df_revenue=pd.read_sql(query,con)
df_revenue

,genre,revenue
0,Rock,826.65
1,Latin,382.14
2,Metal,261.36
3,Alternative & Punk,241.56
4,TV Shows,93.53
5,Jazz,79.20
6,Blues,60.39
7,Drama,57.71
8,R&B/Soul,40.59
9,Classical,40.59


**Q5.** Which artists sell the most?

Iron Maiden is the top-earning artist (about 6% of total revenue).
The top four are all rock or metal acts.

In [63]:
query="""
SELECT ar.Name as ArtistName, SUM(il.UnitPrice * il.Quantity) AS revenue
FROM InvoiceLine il
JOIN Track AS t ON il.TrackId = t.TrackId
JOIN Album AS al ON t.AlbumId = al.AlbumId
JOIN Artist AS ar ON al.ArtistId = ar.ArtistId
GROUP BY ar.Name
ORDER BY revenue DESC
LIMIT 10;
"""
df=pd.read_sql(query,con)
df

,ArtistName,revenue
0,Iron Maiden,138.60
1,U2,105.93
2,Metallica,90.09
3,Led Zeppelin,86.13
4,Lost,81.59
5,The Office,49.75
6,Os Paralamas Do Sucesso,44.55
7,Deep Purple,43.56
8,Faith No More,41.58
9,Eric Clapton,39.60


**Q6.** How does revenue trend month by month?

Monthly revenue is very steady across all 60 months (Jan 2021 to Dec 2025).
The occasional spikes (highest was in Jan 2022) and dips (lowest was in Nov 2023) are small swings around that baseline.

`strftime('%Y-%m', InvoiceDate) AS month`

`strftime` is SQLite’s function for formatting a date. It takes a format pattern and a date column:

`%Y` is the 4-digit year
`%m` is the 2-digit month

In [64]:
query = """
SELECT strftime('%Y-%m', InvoiceDate) AS month, SUM(Total) AS revenue
FROM Invoice
GROUP BY month
ORDER BY month
"""
df_month=pd.read_sql(query,con)
df_month

,month,revenue
0,2021-01,35.64
1,2021-02,37.62
2,2021-03,37.62
3,2021-04,37.62
4,2021-05,37.62
5,2021-06,37.62
6,2021-07,37.62
7,2021-08,37.62
8,2021-09,37.62
9,2021-10,37.62


**`PRAGMA table_info("Album")`** is a special SQLite command that returns details about one table’s columns: name, data type, whether it can be null, and whether it’s a primary key.

In [65]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)

for t in tables['name']:
    cols = pd.read_sql(f'PRAGMA table_info("{t}")', con)
    print(t, '->', list(cols['name']))

Album -> ['AlbumId', 'Title', 'ArtistId']
Artist -> ['ArtistId', 'Name']
Customer -> ['CustomerId', 'FirstName', 'LastName', 'Company', 'Address', 'City', 'State', 'Country', 'PostalCode', 'Phone', 'Fax', 'Email', 'SupportRepId']
Employee -> ['EmployeeId', 'LastName', 'FirstName', 'Title', 'ReportsTo', 'BirthDate', 'HireDate', 'Address', 'City', 'State', 'Country', 'PostalCode', 'Phone', 'Fax', 'Email']
Genre -> ['GenreId', 'Name']
Invoice -> ['InvoiceId', 'CustomerId', 'InvoiceDate', 'BillingAddress', 'BillingCity', 'BillingState', 'BillingCountry', 'BillingPostalCode', 'Total']
InvoiceLine -> ['InvoiceLineId', 'InvoiceId', 'TrackId', 'UnitPrice', 'Quantity']
MediaType -> ['MediaTypeId', 'Name']
Playlist -> ['PlaylistId', 'Name']
PlaylistTrack -> ['PlaylistId', 'TrackId']
Track -> ['TrackId', 'Name', 'AlbumId', 'MediaTypeId', 'GenreId', 'Composer', 'Milliseconds', 'Bytes', 'UnitPrice']


**Q7.** Which employee supports the most customers?

Jane Peacock supports the most customers (21).

The workload is fairly balanced across the three support reps, with only a 3-customer gap between the highest and lowest

In [66]:
query = """
SELECT e.FirstName || ' ' || e.LastName AS employee, COUNT(c.CustomerId) AS customers
FROM Employee e
JOIN Customer c ON c.SupportRepId = e.EmployeeId
GROUP BY e.EmployeeId
ORDER BY customers DESC;
"""
df=pd.read_sql(query,con)
df

,employee,customers
0,Jane Peacock,21
1,Margaret Park,20
2,Steve Johnson,18


**Q8.** Which tracks have never been purchased?

1,519 tracks have never been sold, which is about 43% of the catalog.
So a large share of the catalog has never generated any revenue.

In [67]:
query="""
SELECT t.Name
FROM Track t
LEFT JOIN InvoiceLine il ON t.TrackId = il.TrackId
WHERE il.TrackId IS NULL
"""
df=pd.read_sql(query,con)
df

,Name
0,Let's Get It Up
1,C.O.D.
2,Let There Be Rock
3,Bad Boy Boogie
4,Whole Lotta Rosie
...,...
1514,"Erlkonig, D.328"
1515,"Concerto for Violin, Strings and Continuo in G..."
1516,"L'orfeo, Act 3, Sinfonia (Orchestra)"
1517,"Quintet for Horn, Violin, 2 Violas, and Cello ..."
